# Australia: Coles vs Woolworths on `matchKey`, plus this week's Special Buys

Two runs of
[Australia Supermarket Prices](https://apify.com/yappman/au-supermarket-price-scraper):

1. A search at Coles and Woolworths, compared product by product on `matchKey`.
2. Aldi's Special Buys drops dated this week (Monday to Sunday, Sydney time).

**Cost at Apify Free-tier prices:** at most **$0.25** in total.

| Run | Rows (`maxItems`) | Rows | Start | Most it can cost |
|---|---|---|---|---|
| Search | 60 | $0.12 | $0.005 | $0.125 |
| Special Buys | 60 | $0.12 | $0.005 | $0.125 |

Each row is $0.002 on the Free tier.

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from decimal import Decimal

from pathlib import Path

import numpy as np
import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved, and
    max_total_charge_usd stops the run if its charges would pass the cap.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    print("Charged events:", run.charged_event_counts)
    return run

## 1. Search Coles and Woolworths

`maxItems` is split evenly between the shops, so 60 gives up to 30 each.
Marketplace (third-party) products are left out. Branded products match best,
because `matchKey` includes the brand.

In [ ]:
ACTOR = "yappman/au-supermarket-price-scraper"
QUERIES = ["vegemite", "arnotts tim tam", "cadbury dairy milk"]

# Free tier: 60 rows x $0.002 + 1 start x $0.005 = $0.125
search_run = run_actor(
    ACTOR,
    {"mode": "search", "retailers": ["coles", "woolworths"], "queries": QUERIES,
     "maxItems": 60, "excludeMarketplace": True},
    max_charge_usd=0.13,
)
df = pd.DataFrame(client.dataset(search_run.default_dataset_id).iterate_items())
df = df[df["price"].notna()].copy()
df["effectivePrice"] = df[["price", "loyaltyPrice"]].min(axis=1, skipna=True)
df["onSpecial"] = df["wasPrice"].notna() & (df["wasPrice"] > df["price"])
df.groupby("retailer").size()

## 2. Compare on `matchKey`

When a shop returns more than one row with the same key, the cheapest is used.

In [ ]:
best = (df[df["matchKey"].notna()]
        .sort_values("effectivePrice")
        .drop_duplicates(["matchKey", "retailer"]))

wide = best.pivot(index="matchKey", columns="retailer", values="effectivePrice")
both = wide.dropna(subset=["coles", "woolworths"]).copy()
both["difference"] = (both["coles"] - both["woolworths"]).round(2)
both["cheaper"] = np.select(
    [both["difference"] < 0, both["difference"] > 0], ["coles", "woolworths"], "same"
)
specials = best.pivot(index="matchKey", columns="retailer", values="onSpecial")
both = both.join(specials.add_prefix("special_"))
both = both.join(best.drop_duplicates("matchKey").set_index("matchKey")[["name", "packSize"]])

print(len(both), "products at both;", (both["cheaper"] == "coles").sum(), "cheaper at Coles,",
      (both["cheaper"] == "woolworths").sum(), "cheaper at Woolworths,",
      (both["cheaper"] == "same").sum(), "the same")
both.to_csv(OUTPUT / "au_coles_vs_woolworths.csv")
both.sort_values("difference")

## 3. This week's Aldi Special Buys

The run asks for every date in the current Sydney week. Aldi drops on set days,
and dates Aldi's page does not list are skipped and named in the run's
`RUN_SUMMARY` record.

In [ ]:
today = pd.Timestamp.now(tz="Australia/Sydney").normalize()
monday = today - pd.Timedelta(days=today.weekday())
week = [(monday + pd.Timedelta(days=i)).strftime("%Y-%m-%d") for i in range(7)]

# Free tier: 60 rows x $0.002 + 1 start x $0.005 = $0.125
sb_run = run_actor(
    ACTOR,
    {"mode": "specialBuys", "specialBuysDrops": week, "maxItems": 60},
    max_charge_usd=0.13,
)
sb = pd.DataFrame(client.dataset(sb_run.default_dataset_id).iterate_items())

summary = client.key_value_store(sb_run.default_key_value_store_id).get_record("RUN_SUMMARY")
if summary:
    print("Special Buys summary:",
          summary["value"].get("perRetailer", {}).get("aldi_au", {}).get("specialBuys"))

if sb.empty:
    print("No Special Buys listed for", week[0], "to", week[-1])
else:
    sb["category"] = sb["categoryPath"].apply(lambda p: p[0] if p else None)
    print(sb.groupby("onSaleDate").size().rename("products"))

In [ ]:
if not sb.empty:
    sb.to_csv(OUTPUT / "au_special_buys_this_week.csv", index=False)
    display(sb.sort_values(["onSaleDate", "price"])
              [["onSaleDate", "name", "price", "category", "promotionText"]])

## Notes

* Coles rows carry no rating. Coles barcodes need `colesProductDetails`, which
  costs an extra request per product, so this example matches on `matchKey` only.
* Rows belong to the Actor's data terms, not to this repository's MIT licence.
  The CSVs are written to `output/`, which git ignores.